# MTech Bug-Fix Agent: Colab Feasibility Check

Start every fresh runtime by checking available RAM, disk, and GPU memory. The report recommends a model size before any model weights are downloaded. Follow the gates in `PROJECT_PLAN.md`; do not assume the previous runtime hardware is still available. Then verify one Goldset record and run both matched pilot conditions.

In Colab, choose **Runtime > Change runtime type** and select a GPU if available. Run cells from top to bottom.


In [ ]:
import json
import platform
import shutil
import sys
import torch

def gib(n):
    return round(n / (1024 ** 3), 2)

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
disk_free_gib = gib(shutil.disk_usage('/content').free)
print('Free disk at /content (GiB):', disk_free_gib)
try:
    import psutil
    ram = psutil.virtual_memory()
    ram_total_gib, ram_available_gib = gib(ram.total), gib(ram.available)
    print('System RAM total / available (GiB):', ram_total_gib, '/', ram_available_gib)
except ImportError:
    with open('/proc/meminfo', encoding='utf-8') as memfile:
        meminfo = {line.split(':', 1)[0]: int(line.split()[1]) * 1024 for line in memfile if ':' in line}
    ram_total_gib = gib(meminfo['MemTotal'])
    ram_available_gib = gib(meminfo.get('MemAvailable', meminfo['MemFree']))
    print('System RAM total / available (GiB), from /proc/meminfo:', ram_total_gib, '/', ram_available_gib)
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available. Select a Colab GPU runtime, rerun this resource check, and only then load a model.')
props = torch.cuda.get_device_properties(0)
gpu_total_gib = gib(props.total_memory)
gpu_free_gib = gib(torch.cuda.mem_get_info(0)[0])
print('GPU:', props.name)
print('GPU memory total / currently free (GiB):', gpu_total_gib, '/', gpu_free_gib)
print('Python executable:', sys.executable)

# Thresholds preserve download/cache and inference headroom; see PROJECT_PLAN.md.
if gpu_free_gib >= 11 and disk_free_gib >= 20:
    MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
    PILOT_MAX_INPUT_TOKENS, PILOT_MAX_NEW_TOKENS = 3072, 768
elif gpu_free_gib >= 6 and disk_free_gib >= 12:
    MODEL_ID = 'Qwen/Qwen2.5-Coder-3B-Instruct'
    PILOT_MAX_INPUT_TOKENS, PILOT_MAX_NEW_TOKENS = 2048, 512
elif gpu_free_gib >= 4.5 and disk_free_gib >= 8:
    MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
    PILOT_MAX_INPUT_TOKENS, PILOT_MAX_NEW_TOKENS = 1536, 384
else:
    raise RuntimeError('Available GPU memory or disk is below the pilot minimum. Select a larger Colab GPU/runtime before downloading model weights.')
RESOURCE_REPORT = {
    'python': platform.python_version(), 'torch': torch.__version__,
    'disk_free_gib': disk_free_gib, 'ram_total_gib': ram_total_gib,
    'ram_available_gib': ram_available_gib, 'gpu': props.name,
    'gpu_total_gib': gpu_total_gib, 'gpu_free_gib': gpu_free_gib,
    'model_id': MODEL_ID, 'max_input_tokens': PILOT_MAX_INPUT_TOKENS,
    'max_new_tokens': PILOT_MAX_NEW_TOKENS,
}
print('Resource-based model recommendation:', MODEL_ID)
print('Pilot input/output token limits:', PILOT_MAX_INPUT_TOKENS, '/', PILOT_MAX_NEW_TOKENS)
print('Resource report JSON:', json.dumps(RESOURCE_REPORT, indent=2))

## Download and install the project

The repository is public. This cell clones it into Colab's temporary storage and installs the lightweight project package. Colab storage resets when its runtime is discarded, so rerun this notebook after a reset.

In [61]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/pisalaarti05/mtech-bug-fix-agent.git'
PROJECT_DIR = Path('/content/mtech-bug-fix-agent')

if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--editable', str(PROJECT_DIR)], check=True)
print('Project installed from:', PROJECT_DIR)

Project installed from: /content/mtech-bug-fix-agent


## Confirm the command is ready

This displays the program's options. A successful result means Colab can import the project. Do not choose a model yet; first share the runtime details above.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'bugfix_agent.cli', '--help'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'bugfix_agent.cli', '--help'], returncode=0)

## Check one SWE-bench Lite task and Colab capacity

SWE-bench Lite is a suitable issue-driven candidate, but its official evaluation runs Docker containers and needs at least 120 GB of free disk. This cell only reads one task record and checks your current Colab resources; it does not reveal the reference patch or run benchmark tests. If Docker or disk is insufficient, do not claim official SWE-bench scores from this runtime.

In [ ]:
import shutil
import subprocess
import sys

# Read one benchmark example without downloading the entire dataset.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets>=3.0'], check=True)
from datasets import load_dataset

tasks = load_dataset('princeton-nlp/SWE-bench_Lite', split='test', streaming=True)
task = next(iter(tasks))
print('Instance:', task['instance_id'])
print('Repository:', task['repo'])
print('Base commit:', task['base_commit'])
print('Issue description (first 1800 chars):')
print(task['problem_statement'][:1800])

disk = shutil.disk_usage('/content')
print(f'Free Colab disk: {disk.free / (1024**3):.1f} GiB')
docker = subprocess.run(['bash', '-lc', 'docker --version'], capture_output=True, text=True)
print('Docker available:', docker.returncode == 0)
if docker.returncode == 0:
    print(docker.stdout.strip())
else:
    print('Docker is not available in this runtime.')
print('Official SWE-bench harness minimum free disk: 120 GB.')


Instance: astropy__astropy-12907
Repository: astropy/astropy
Base commit: d16bfe05a744909de4b27f5875fe0d4ed41ce607
Issue description (first 1800 chars):
Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels
Consider the following model:

```python
from astropy.modeling import models as m
from astropy.modeling.separable import separability_matrix

cm = m.Linear1D(10) & m.Linear1D(5)
```

It's separability matrix as you might expect is a diagonal:

```python
>>> separability_matrix(cm)
array([[ True, False],
       [False,  True]])
```

If I make the model more complex:
```python
>>> separability_matrix(m.Pix2Sky_TAN() & m.Linear1D(10) & m.Linear1D(5))
array([[ True,  True, False, False],
       [ True,  True, False, False],
       [False, False,  True, False],
       [False, False, False,  True]])
```

The output matrix is again, as expected, the outputs and inputs to the linear models are separable and independent of each other.

If however,

## Verify one record from the smaller Goldset benchmark

Goldset provides real Python bug records, project commit IDs, license metadata, and regression tests. This cell installs the pinned benchmark source and selects one recent function-level task so the code is small enough for the Colab T4. It verifies that the recorded test fails before the fix and passes after it.

The verifier clones a public repository and runs its test code in this Colab session. It does not use Docker. A record may be reported as skipped if its dependencies or tests cannot run in the available environment.

In [62]:
from pathlib import Path
import json
import subprocess
import sys

GOLDSET_DIR = Path('/content/goldset')
GOLDSET_REVISION = 'fe4f9bfbcb46637481242af2315c6b59951710b4'
if not GOLDSET_DIR.exists():
    subprocess.run(['git', 'clone', '--quiet', '--no-checkout', 'https://github.com/andysalvo/goldset.git', str(GOLDSET_DIR)], check=True)
subprocess.run(['git', 'fetch', '--quiet', '--depth', '1', 'origin', GOLDSET_REVISION], cwd=GOLDSET_DIR, check=True)
subprocess.run(['git', 'checkout', '--quiet', 'FETCH_HEAD'], cwd=GOLDSET_DIR, check=True)

# Remove a possibly preinstalled, unrelated package with the same name, then install this pinned benchmark source.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'goldset'], capture_output=True, text=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(GOLDSET_DIR), 'uv'], check=True)

# Clear a stale import if this cell is being rerun in the same notebook session.
for module_name in list(sys.modules):
    if module_name == 'goldset' or module_name.startswith('goldset.'):
        del sys.modules[module_name]
import importlib
goldset_source = str(GOLDSET_DIR)
if goldset_source in sys.path:
    sys.path.remove(goldset_source)
sys.path.insert(0, goldset_source)
importlib.invalidate_caches()
import goldset
import goldset.exec
runner_path = Path(goldset.exec.__file__).resolve()
if not runner_path.is_relative_to(GOLDSET_DIR.resolve()):
    raise RuntimeError(f'Goldset helpers loaded from the wrong location: {runner_path}')
print('Goldset runner module:', runner_path)

records = [json.loads(line) for line in (GOLDSET_DIR / 'oracles.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
candidates = [r for r in records if r.get('granularity') == 'function' and r.get('committed_at', '') >= '2025-01-01']
if not candidates:
    raise RuntimeError('No function-level Goldset records dated 2025-01-01 or later were found.')
record = sorted(candidates, key=lambda r: (r['committed_at'], r['repo'], r['fingerprint']))[0]
goldset_sha = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=GOLDSET_DIR, check=True, capture_output=True, text=True).stdout.strip()
print('Goldset repository revision:', goldset_sha)
print('Record:', record['fingerprint'])
print('Fix date:', record['committed_at'])
print('Project:', record['repo'])
print('Project license:', record['license'])
print('Bug specification:', record['spec'][:1200])
print('Running one-record verification now...')
verification = subprocess.run([sys.executable, 'verify.py', '--corpus', 'oracles.jsonl', '--fingerprint', record['fingerprint']], cwd=GOLDSET_DIR, capture_output=True, text=True)
print(verification.stdout)
if verification.stderr.strip():
    print('Verifier messages:', verification.stderr[-3000:])
print('Verifier process exit code:', verification.returncode)
if verification.returncode != 0:
    raise RuntimeError('Goldset verifier reported a failed record. Read the verifier output above.')
if 'passed 1' not in verification.stdout and 'skipped 1' not in verification.stdout:
    raise RuntimeError('Could not find a clear pass/skip summary in the verifier output.')


Goldset runner module: /content/goldset/goldset/exec.py
Goldset repository revision: fe4f9bfbcb46637481242af2315c6b59951710b4
Record: cbafae4011f8117b
Fix date: 2025-01-02
Project: google/etils
Project license: Apache-2.0
Bug specification: Like `Enum`, but `enum.auto()` assigns `str` rather than `int`.

``python
class MyEnum(epy.StrEnum):
  SOME_ATTR = enum.auto()
  OTHER_ATTR = enum.auto()

assert MyEnum('some_attr') is MyEnum.SOME_ATTR
assert MyEnum.SOME_ATTR == 'some_attr'
``

`StrEnum` is case insensitive.
Running one-record verification now...
verifying 1 record(s)

[1/1] pass  google/etils  StrEnum

passed 1   failed 0   skipped 0

Verifier messages: This clones public projects and runs their test suites on this machine. Use a container if that matters to you.


Verifier process exit code: 0


## Model choice for this runtime

The resource check selects a model tier using current free GPU memory and disk. Do not change the model between paired conditions. If loading runs out of memory, record the error, restart, rerun the resource check, and use the next smaller tier.

In [ ]:
import subprocess
import sys

# Install only the model libraries. Keep Colab's existing PyTorch/CUDA build.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
    'transformers', 'accelerate', 'bitsandbytes'
], check=True)
from importlib.metadata import version
for package in ('transformers', 'accelerate', 'bitsandbytes'):
    print(f'{package}:', version(package))
print('Model libraries installed.')

transformers: 5.22.1
accelerate: 1.15.0
bitsandbytes: 0.50.2
Model libraries installed.


## Load the selected model and make one short request

This uses 4-bit NF4 compression. The model ID and pilot limits come from the fresh runtime resource report above. The same loaded model is reused for both conditions.

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_DIR = Path('/content/mtech-bug-fix-agent')
SOURCE_DIR = PROJECT_DIR / 'src'
if not (SOURCE_DIR / 'bugfix_agent' / 'model.py').is_file():
    raise FileNotFoundError('Run the Download and install the project cell first.')
sys.path.insert(0, str(SOURCE_DIR))
from bugfix_agent.model import HuggingFaceModel

if 'MODEL_ID' not in globals():
    raise RuntimeError('Run the resource check cell first; it recommends a model using current free GPU memory and disk.')
if (
    'model' in globals()
    and isinstance(model, HuggingFaceModel)
    and model.model_id == MODEL_ID
    and getattr(model, '_model', None) is not None
):
    print('Reusing the model already loaded in this Colab runtime.')
else:
    import gc
    if 'model' in globals():
        del model
    gc.collect()
    torch.cuda.empty_cache()
    model = HuggingFaceModel(MODEL_ID, max_new_tokens=min(PILOT_MAX_NEW_TOKENS, 128), max_input_tokens=min(PILOT_MAX_INPUT_TOKENS, 1024))
torch.cuda.reset_peak_memory_stats()
print('Model loaded:', MODEL_ID)
print('Model snapshot:', model.revision)
print('Load time (seconds):', model.load_seconds)
print('GPU memory after load (GiB):', round(torch.cuda.memory_allocated() / (1024 ** 3), 2))

prompt = (
    'Return only a unified diff for this small Python bug. '
    'Current file: def divide(a, b): return a / b. '
    'Bug: when b is zero, return None instead of raising ZeroDivisionError.'
)
response = model.generate(prompt)
print('Generated response:')
print(response)
print('Token and time record:', model.last_usage)
print('Peak GPU memory (GiB):', round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2))
if not response.strip():
    raise RuntimeError('The model loaded but returned an empty response.')

## Run one Goldset task through the agent with and without retrieval

This runs the same small function-level task twice: first with only the benchmark specification and buggy function, then with up to three additional code chunks retrieved from the buggy repository. The target source file and test file are excluded from retrieval. The test and reference fix are used only to validate the environment and score the generated replacement; they are not included in the prompt.

The model from the previous cell is reused. The shorter input and output limits reduce GPU memory use for the T4. The task is still scored with the same regression test in both conditions.

In [ ]:
import json
from pathlib import Path
import torch
from bugfix_agent.benchmark import run_goldset_record

if 'model' not in globals() or 'record' not in globals():
    raise RuntimeError('Run the Goldset verification cell and the model load cell first.')

# The model pilot used a short output limit. Raise it for complete function/file replacements.
model.max_new_tokens = PILOT_MAX_NEW_TOKENS
model.max_input_tokens = PILOT_MAX_INPUT_TOKENS
torch.cuda.empty_cache()
run_dir = PROJECT_DIR / 'artifacts' / 'runs' / 'goldset_pilot'
pilot_records = []
resource_path = run_dir / 'resource_report.json'
resource_path.parent.mkdir(parents=True, exist_ok=True)
resource_path.write_text(json.dumps(RESOURCE_REPORT, indent=2), encoding='utf-8')
print('Saved resource report:', resource_path)
print('Using model:', MODEL_ID, '| input:', PILOT_MAX_INPUT_TOKENS, '| output:', PILOT_MAX_NEW_TOKENS)
for retrieval_enabled in (False, True):
    torch.cuda.empty_cache()
    result_path = run_goldset_record(
        record,
        model,
        run_dir,
        use_retrieval=retrieval_enabled,
        top_k=3,
        corpus_revision=goldset_sha,
    )
    result = json.loads(result_path.read_text(encoding='utf-8'))
    pilot_records.append(result)
    print(('Retrieval' if retrieval_enabled else 'No retrieval') + ':', result['test_status'])
    if result.get('skip_reason'):
        print('Reason:', result['skip_reason'])
    if result.get('failure_reason'):
        print('Reason:', result['failure_reason'])
    print('Buggy revision test:', result.get('buggy_revision_test_status', 'not reached'))
    print('Reference fix test:', result.get('reference_fix_test_status', 'not reached'))
    print('Generation attempted:', 'model_response' in result)
    if result.get('model_response') and result['test_status'] != 'passed':
        print('Generated replacement preview:')
        print(result['model_response'][:1500])
    print('Run record:', result_path)
print('Model revision:', model.revision)
print('Both conditions finished. These one-task results are feasibility evidence, not a benchmark score.')
